In [1]:
! pip install --upgrade mysql-connector-python

In [3]:
import cv2
import mysql.connector as mysql  # Make sure to run: pip install mysql-connector-python
import time

# --- Database Configuration ---
db_config = {
    "host": "localhost",
    "user": "root",        # Replace with your MySQL username
    "password": "@akankhya9090",        # Replace with your MySQL password
    "database": "attendance_db"
}

# Connect to Database and Create Table
try:
    conn = mysql.connect(**db_config)
    cursor = conn.cursor()
    
    # Create Table if it doesn't exist
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS student_photos (
            id INT AUTO_INCREMENT PRIMARY KEY,
            student_name VARCHAR(100) NOT NULL,
            image_index INT NOT NULL,
            photo LONGBLOB NOT NULL,
            uploaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    print("Database connection successful and table verified.")
except mysql.Error as err:
    print(f"Database Error: {err}")
    exit()

# --- Face Capture Setup ---
student_name = input("Enter Student Name: ")

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)
count = 0

print("Capturing 20 face images and saving to database...")
print("Look at the camera and slightly change your pose.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Camera Error!")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(50, 50)
    )

    for (x, y, w, h) in faces:
        # Capture original color face & resize
        face = frame[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100))

        count += 1

        # 1. Compress face frame to JPEG format in memory
        success, encoded_image = cv2.imencode('.jpg', face)
        
        if success:
            # 2. Convert encoded image to raw bytes
            binary_data = encoded_image.tobytes()
            
            # 3. Insert into MySQL database
            try:
                query = """
                    INSERT INTO student_photos (student_name, image_index, photo) 
                    VALUES (%s, %s, %s)
                """
                cursor.execute(query, (student_name, count, binary_data))
                conn.commit()
            except mysql.Error as e:
                print(f"\nFailed to upload image {count}: {e}")
                count -= 1 # Reset count for failed attempt

        # Draw rectangle and HUD overlay
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"Captured & Uploaded: {count}/20",
            (10, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.imshow("Face Capture", frame)
        time.sleep(0.5)  # Delay between shots

        if count >= 20:
            break

    cv2.imshow("Face Capture", frame)

    if count >= 20:
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
cursor.close()
conn.close()

print(f"\nSuccess! 20 images saved safely in database 'attendance_db' for {student_name}.")

ModuleNotFoundError: No module named 'mysql'